# OpenAI Helpers Pipelines Demo

This notebook shows the current wrapper behavior for schema hints, tool calls, logging, and chat sessions against a local OpenAI-compatible server.

Default endpoint: `http://127.0.0.1:8080/v1`.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import asdict
from pathlib import Path

from openai import OpenAI
from openai_helpers_piplines_package import (
    JsonFixPipeline,
    LoggerPipeline,
    LoopGuardPipeline,
    ToolPipeline,
    chat_session,
    dict_to_pydantic_schema,
    with_pipelines,
)
from pipelines.chat import PipelineRequestError

BASE_URL = os.environ.get('OPENAI_HELPERS_BASE_URL', 'http://127.0.0.1:8080/v1')
API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed-for-local-server')
MODEL_NAME = os.environ.get('OPENAI_HELPERS_MODEL', '')
LOG_PATH = os.environ.get('OPENAI_HELPERS_LOG', 'demo_pipeline.log')

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
logger_layer = LoggerPipeline(path=LOG_PATH)
json_chat = with_pipelines(client.chat.completions, layers=[logger_layer, LoopGuardPipeline(), JsonFixPipeline()])
tool_chat = with_pipelines(client.chat.completions, layers=[logger_layer, ToolPipeline(), LoopGuardPipeline(), JsonFixPipeline()])
combined_chat = with_pipelines(client.chat.completions, layers=[logger_layer, ToolPipeline(), LoopGuardPipeline(), JsonFixPipeline()])

models_response = client.models.list()
available_models = [model.id for model in models_response.data]
model_name = MODEL_NAME or (available_models[0] if available_models else None)
if not model_name:
    raise RuntimeError('No model found and OPENAI_HELPERS_MODEL was not provided')

print('Selected model:', model_name)
print('Log file:', Path(LOG_PATH).resolve())


In [ ]:
schema = {
    'title': str,
    'score': float,
    'tags': [str],
    'meta': {
        'source': str,
        'priority': int,
    },
}

model_cls = dict_to_pydantic_schema(schema)
print(json.dumps(model_cls.model_json_schema(), indent=2, ensure_ascii=False)[:1000])


In [ ]:
structured_result = await json_chat.create(
    model=model_name,
    messages=[
        {'role': 'user', 'content': 'Return one compact JSON object for a local model helper demo.'},
    ],
    temperature=0.2,
    max_tokens=500,
    schema_dict=schema,
    return_trace=True,
)

print('parsed:')
print(json.dumps(structured_result.parsed, indent=2, ensure_ascii=False))
print('trace:')
for event in structured_result.trace:
    print(f'{event.level}/{event.action}', event.detail)


In [ ]:
def add(a: int, b: int) -> dict[str, int]:
    return {'sum': a + b}

tool_result = await tool_chat.create(
    model=model_name,
    messages=[
        {'role': 'system', 'content': 'Use the add tool when arithmetic is requested.'},
        {'role': 'user', 'content': 'Use the add tool to compute 12 + 30, then explain the result in one short sentence.'},
    ],
    temperature=0.1,
    max_tokens=500,
    tool_sources=[{'add': add}],
    return_trace=True,
)

print('assistant:', tool_result.response['choices'][0]['message'].get('content', ''))
print('tool executions:')
print(json.dumps([asdict(execution) for execution in tool_result.tool_executions], indent=2, ensure_ascii=False))
print('trace:')
for event in tool_result.trace:
    print(f'{event.level}/{event.action}', event.detail)


In [ ]:
combined_result = await combined_chat.create(
    model=model_name,
    messages=[
        {'role': 'system', 'content': 'Use the add tool for arithmetic. Final answer must match the JSON schema.'},
        {'role': 'user', 'content': 'Call the add tool to compute 7 + 8, then return JSON with answer and method.'},
    ],
    temperature=0.1,
    max_tokens=600,
    tool_sources=[{'add': add}],
    schema_dict={'answer': str, 'method': str},
    return_trace=True,
)

print('parsed:')
print(json.dumps(combined_result.parsed, indent=2, ensure_ascii=False))
print('tool executions:', [asdict(execution) for execution in combined_result.tool_executions])
print('trace events:', [f'{event.level}/{event.action}' for event in combined_result.trace])


In [ ]:
session = chat_session(
    combined_chat,
    model=model_name,
    return_trace=True,
)

session.messages.append_role_content({
    'system': 'Use the add tool for arithmetic. Final answer must match the JSON schema.',
})

session_result = await session.step(
    role_content={'user': 'Call the add tool to compute 3 + 4, then return JSON with answer and method.'},
    max_tokens=500,
    schema_dict={'answer': str, 'method': str},
)

print('session parsed:')
print(json.dumps(session_result.parsed, indent=2, ensure_ascii=False))
print('session messages:', len(session.messages))


## Error Handling

The notebook uses `attempt(...)` for expected request failures so they can be handled as values while real bugs still raise.


In [ ]:
async def attempt(coro):
    try:
        return await coro
    except PipelineRequestError as error:
        return error

dead_chat = with_pipelines(
    OpenAI(base_url='http://127.0.0.1:1/v1', api_key='x', max_retries=0, timeout=3.0).chat.completions,
    layers=[ToolPipeline()],
)

expected = await attempt(
    dead_chat.create(
        model=model_name,
        messages=[{'role': 'user', 'content': 'Say hello in one short sentence.'}],
        max_tokens=200,
    )
)

if isinstance(expected, PipelineRequestError):
    print('expected failure captured as value:')
    print(expected)
else:
    print('unexpected success:', expected.response['choices'][0]['message']['content'])


## Notes

- `PipelineRequestError` is the main request-failure type.
- Structured output repairs are graceful: successful fallback still returns `result.parsed`.
- `attempt(...)` is the notebook pattern for expected request failures as values.
- `LoggerPipeline` writes request, stream, tool, trace, and end-of-call metadata to `LOG_PATH`.
- `LoopGuardPipeline` watches the streamed generation while the model is producing tokens.
- Use top-level `await` in Jupyter cells; do not wrap these calls with `asyncio.run(...)`.


In [ ]:
log_file = Path(LOG_PATH)
print('Log preview:', log_file.resolve())
if log_file.exists():
    log_text = log_file.read_text(encoding='utf-8')
    print('request count:', log_text.count('pipeline.chat: request'))
    print('thinking markers:', log_text.count('[THINKING]'))
    print('message markers:', log_text.count('[MESSAGE]'))
    print('tool result blocks:', log_text.count('|TOOL-RESULT|{'))
    print(log_text[-2000:])
else:
    print('Log file was not created')
